## Deployment Configurations
A minimal deployment configuration looks like:
```yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: basic-deployment
  labels:
    app: web
spec:
  replicas: 3
  selector:
    matchLabels:
      app: web
  template:
    metadata:
      labels:
        app: web
    spec:
      containers:
        - name: nginx
          image: nginx:1.25
          ports:
            - containerPort: 80
```

How does a deployment remember the pods it spawned? It uses the `spec.selector` attribute. This would match the label on the pod - the rule is that `spec.selector.matchLabels` matches `spec.template.metadata.labels`.

The `template` section defines how a pod is created. Its values can essentially be the same as a dedicated pod configuration. The only difference we observe are:
- We omit `template.metadata.name` in deployment, whereas it is required in pod
- We omit `template.metadata.namespace` in deployment since it inherits from the deployment's namespace
- In `template.spec.restartPolicy` is always set as `Always`

To achieve a zero-downtime rolling update:
```yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: rolling-deployment
spec:
  replicas: 4

  strategy:
    type: RollingUpdate
    rollingUpdate:
      maxSurge: 25%        # Can spin up 25% extra pods above desired replica count
      maxUnavailable: 0%   # Guarantees 100% capacity during an update

  selector:
    matchLabels:
      app: api-service
  template:
    metadata:
      labels:
        app: api-service
    spec:
      containers:
        - name: api
          image: api:v2.0.0
          readinessProbe:
            httpGet:
              path: /ready
              port: 8080
            initialDelaySeconds: 5
            periodSeconds: 5
```

The image below illustrates how rolling update works. At any given time the number of replicas can go `maxSurge` percentage above the limit. Also `maxUnavailable` is the range the pod count can go below the configured limit while new pods of new version are coming up.  
![rolling_udpate](./images/zero_downtime.png)

This strategy also explain why readiness probe is needed. Without a readiness probe, Kubernetes considers a new v2 pod ready the exact second its process launches. It will immediately kill healthy v1 pods before v2 has even finished connecting to the database or loading into memory. A readiness probe guarantees traffic stays on v1 until v2 responds with HTTP 200.

**Horizontal Pod Scaling:** to define autoscaling rules, the pods should declare its resource requirement:
```yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: order-api
# ...
spec:
  containers:
    - name: order-api
      image: my-registry/order-api:1.4.2
      ports:
        - containerPort: 8080

      resources:
        requests:
          cpu: "250m"
          memory: "256Mi"
        limits:
          cpu: "500m"
          memory: "512Mi"
```

Then create a autoscaler:
```yaml
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: order-api-hpa
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: order-api  # Must match the Deployment's `metadata.name` exactly
  minReplicas: 2
  maxReplicas: 10
  metrics:
    - type: Resource
      resource:
        name: cpu
        target:
          type: Utilization
          averageUtilization: 80
  behavior:  # Optional
    scaleDown:
      stabilizationWindowSeconds: 300  # Wait 5 min of sustained low usage before scaling down
```

Once a HPA is active, it owns the replica count (and not the Deployment).